# Document Summariser — CUAD Legal Contract Pipeline

An end-to-end pipeline for extracting key clauses and generating summaries from legal contracts, built for the CUAD (Contract Understanding Atticus Dataset) subset.

**Pipeline overview:**
1. **Data Loading & Preprocessing** — load a 50-contract subset of CUAD, extract text from each PDF (with OCR fallback for scanned pages), clean the text, and build a manifest linking extracted files back to their source contracts.
2. **LLM-Powered Extraction & Summarization** — use an LLM (DeepSeek V4 Flash) to extract termination, confidentiality, and liability clauses, plus a 100–150 word summary, per contract.
3. **Ground-Truth Evaluation** — validate clause-presence detection against CUAD's own expert annotations (`master_clauses.csv`).

See the accompanying README for setup instructions, the solution flow diagram, and a discussion of results and known limitations.

## Part 1 — Data Loading & Preprocessing

### 1.1 Install dependencies

In [ ]:
# Workspace dependencies.
%pip install numpy pandas matplotlib seaborn scikit-learn datasets huggingface pdfplumber paddleocr pdf2image pillow paddlepaddle openai

### 1.2 Imports

In [ ]:
import unicodedata
import re
import random
import os
import json

import numpy as np
import pandas as pd
import pdfplumber
import pdf2image
import PIL
from datasets import load_dataset
from paddleocr import PaddleOCR
from openai import OpenAI

### 1.3 Load a 50-contract subset of CUAD

In [ ]:
# Loading a subset of the CUAD dataset (Part I) for demonstration purposes.
# The full dataset can be loaded by removing the `data_dir` argument.
dataset = load_dataset("theatticusproject/cuad", data_dir="CUAD_v1/full_contract_pdf/Part_I")

# Randomly select 50 indices from the training set.
indices = random.sample(range(len(dataset["train"])), 50)

# Build a subset of the dataset using the selected indices.
# This is a list of dictionaries, each containing the 'pdf' and 'label' keys.
# NOTE: 'label' here is the contract CATEGORY (e.g. Co_Branding, Distributor),
# not clause-level ground truth. Clause-level ground truth is loaded separately
# in Part 3 from CUAD's master_clauses.csv.
subset = [dataset["train"][i] for i in indices]

### 1.4 Initialize the OCR model

Initialized once globally to avoid the overhead of re-creating the model for every page. `enable_mkldnn=False` disables oneDNN acceleration, which is required for stable CPU inference on this PaddlePaddle version.

In [ ]:
ocr = PaddleOCR(use_textline_orientation=True, lang="en", enable_mkldnn=False)

### 1.5 PDF extraction and text-cleaning helpers

In [ ]:
def page_to_image(page_object, dpi=300):
    """
    Converts a PDF page object to an image using pdf2image.

    Args:
        page_object: A PDF page object from the CUAD dataset.
        dpi: The resolution of the output image in dots per inch (default is 300).

    Returns:
        A NumPy array representing the converted page image.
    """
    image = page_object.to_image(resolution=dpi)
    img_array = np.array(image.original)
    return img_array


def extract_text_from_image(image_array):
    """
    Extracts text from an image using PaddleOCR.

    Args:
        image_array: A NumPy array representing a page of the PDF.

    Returns:
        A string containing the extracted text from the image.
    """
    result = ocr.predict(image_array)
    text = "\n".join([line[1][0] for line in result[0]])
    return text


def extract_text_from_pdf(pdf_object):
    """
    Extracts text from a PDF object using pdfplumber, falling back to OCR for pages
    where the text layer is missing or too short to be useful.

    Args:
        pdf_object: A PDF object from the CUAD dataset.

    Returns:
        A string containing the extracted text from the PDF.
    """
    text = []
    for page in range(len(pdf_object.pages)):
        extracted_text = pdf_object.pages[page].extract_text(x_tolerance=1.5, y_tolerance=3)
        if extracted_text is None or len(extracted_text.strip()) < 10:
            image = page_to_image(pdf_object.pages[page])
            ocr_extracted_text = extract_text_from_image(image)
            text.append(ocr_extracted_text)
        else:
            text.append(extracted_text)
    return "\n".join(text)


def clean_text(text):
    """
    Cleans extracted text by normalizing Unicode, replacing non-breaking spaces,
    normalizing line endings, removing extra spaces, and removing excessive blank lines.

    Args:
        text: A string containing the extracted text from the PDF.

    Returns:
        A cleaned string with normalized Unicode, replaced non-breaking spaces,
        normalized line endings, removed extra spaces, and removed excessive blank lines.
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00A0", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def process_pdf(pdf_object):
    """
    Processes a PDF object by extracting and cleaning its text.

    Args:
        pdf_object: A PDF object from the CUAD dataset.

    Returns:
        A cleaned string containing the extracted text from the PDF.
    """
    extracted_text = extract_text_from_pdf(pdf_object)
    cleaned_text = clean_text(extracted_text)
    return cleaned_text


def create_textfile_from_pdf(pdf_object, output_path):
    """
    Creates a text file from a PDF object by extracting and cleaning its text.

    Args:
        pdf_object: A PDF object from the CUAD dataset.
        output_path: The path where the text file will be saved.

    Returns:
        None. The function saves the cleaned text to a text file at the specified output path.
    """
    cleaned_text = process_pdf(pdf_object)
    with open(output_path, "w", encoding="utf-8") as text_file:
        text_file.write(cleaned_text)

### 1.6 Process all 50 contracts and build the manifest

Saves one cleaned `.txt` file per contract, plus a `manifest.json` mapping each extracted file back to its CUAD source filename, category, and dataset index. This manifest is the join key used in Part 3 for ground-truth evaluation.

In [ ]:
# Processing our PDFs in subset list.
os.makedirs("extracted_pdfs", exist_ok=True)

manifest = []
for pdf in range(len(subset)):
    create_textfile_from_pdf(
        subset[pdf]["pdf"],
        f"extracted_pdfs/pdf_{pdf}.txt"
    )

    pdf_path = subset[pdf]["pdf"].stream.name
    category = os.path.basename(os.path.dirname(pdf_path))
    filename = os.path.basename(pdf_path)

    manifest.append({
        "contract_id": f"pdf_{pdf}",
        "dataset_index": indices[pdf],
        "category": category,
        "source_filename": filename,
        "label": subset[pdf]["label"],
    })

with open("extracted_pdfs/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Successfully extracted text and saved manifest")

## Part 2 — LLM-Powered Clause Extraction & Summarization

### 2.1 LLM client, prompt template, and extraction function

Uses DeepSeek V4 Flash via an OpenAI-compatible API endpoint. The full, untruncated contract text is passed to the model — truncating input meaningfully reduces accuracy (a liability clause correctly identified with the full ~43,000-character contract was missed entirely at a 6,000-character truncation). `max_chars=200_000` is a generous safety ceiling only, well above any CUAD contract length.

`max_tokens=4000` is set generously to account for the model's internal reasoning tokens, which are consumed before the final JSON answer is produced; a smaller budget was found to be fully consumed by reasoning on longer contracts, resulting in empty output.

**API key handling:** the key is read from the `DEEPSEEK_API_KEY` environment variable rather than hardcoded, since this repository is public.

In [ ]:
client = OpenAI(
    base_url="https://api.aicredits.in/v1",
    api_key=os.environ.get("DEEPSEEK_API_KEY", "API_KEY"),  # set via environment variable
)

MODEL_NAME = "deepseek/deepseek-v4-flash"

CLAUSE_SCHEMA_HINT = {
    "termination_clause": "string",
    "confidentiality_clause": "string",
    "liability_clause": "string",
    "summary": "string",
}

PROMPT_TEMPLATE = """You are a legal analyst reviewing a contract. Read the contract text below and extract the following information.

For each clause type, quote or closely paraphrase the exact relevant text from the contract. If a clause type genuinely does not appear anywhere in the contract, respond with exactly "Not found in contract" for that field — do not invent or infer content that isn't present.

1. termination_clause: Conditions under which either party may terminate the agreement.
2. confidentiality_clause: Obligations regarding confidential information, non-disclosure, or trade secrets.
3. liability_clause: Limitations of liability, indemnification, or damages provisions.
4. summary: A 100-150 word summary covering (a) the purpose of the agreement, (b) key obligations of each party, (c) notable risks or penalties.

Respond with ONLY a valid JSON object in exactly this shape, no markdown fences, no extra commentary:
{schema}

Contract text:
---
{contract_text}
---
"""

def _strip_markdown_fences(raw: str) -> str:
    """Some models wrap JSON in ```json ... ``` despite instructions not to."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        raw = raw.replace("json", "", 1).strip()
    return raw


def extract_contract_info(contract_text: str, model: str = MODEL_NAME, max_chars: int = 200_000) -> dict:
    """
    Sends a single contract's text to the LLM and returns structured clause
    extraction + summary as a dict.

    Args:
        contract_text: The cleaned, full text of a single contract.
        model: The model identifier to call.
        max_chars: Safety ceiling on input length (not an active truncation
            point for any real CUAD contract; see note above on why
            truncation should be avoided).

    Returns:
        A dict with keys: termination_clause, confidentiality_clause,
        liability_clause, summary. Falls back to "PARSE_ERROR" values if the
        model's response isn't valid JSON.
    """
    truncated_text = contract_text[:max_chars]
    prompt = PROMPT_TEMPLATE.format(
        schema=json.dumps(CLAUSE_SCHEMA_HINT, indent=2),
        contract_text=truncated_text,
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=4000,
    )
    raw = response.choices[0].message.content
    print(f"  -> {len(truncated_text)} chars in, {len(raw)} chars out")

    cleaned = _strip_markdown_fences(raw)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print("  !! Failed to parse JSON. Raw output:")
        print(raw[:500])
        return {
            "termination_clause": "PARSE_ERROR",
            "confidentiality_clause": "PARSE_ERROR",
            "liability_clause": "PARSE_ERROR",
            "summary": "PARSE_ERROR",
        }

### 2.2 Full pipeline runner

In [ ]:
def run_pipeline(manifest_path="extracted_pdfs/manifest.json", text_dir="extracted_pdfs"):
    """
    Runs extraction across every contract in the manifest and saves all
    results to results.json. Individual contract failures are caught and
    recorded as "ERROR" rows so one bad contract doesn't abort the whole run.
    """
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)

    results = []
    for i, entry in enumerate(manifest):
        contract_id = entry["contract_id"]
        text_path = f"{text_dir}/{contract_id}.txt"
        with open(text_path, encoding="utf-8") as f:
            contract_text = f.read()

        print(f"[{i+1}/{len(manifest)}] Processing {contract_id}...")

        try:
            info = extract_contract_info(contract_text)
        except Exception as e:
            print(f"  !! Failed on {contract_id}: {e}")
            info = {
                "termination_clause": "ERROR",
                "confidentiality_clause": "ERROR",
                "liability_clause": "ERROR",
                "summary": "ERROR",
            }

        results.append({"contract_id": contract_id, **info})

    with open("results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"\nDone. {len(results)} contracts processed, saved to results.json")
    return results

### 2.3 Diagnostic utility (optional)

A supplementary tool for inspecting individual contract responses in detail — raw model output, `finish_reason`, and token usage — rather than the summarized pass/fail view `run_pipeline` provides. Not required for the main pipeline; useful for troubleshooting a specific contract if needed.

In [ ]:
def debug_run(manifest_path="extracted_pdfs/manifest.json", text_dir="extracted_pdfs",
              contract_ids=None, model=MODEL_NAME, max_chars=200_000):
    """
    Runs extraction on a small, specific set of contracts and prints full
    diagnostic detail for each — raw response, finish_reason, char counts,
    and any parse errors — instead of silently swallowing failures.

    Args:
        contract_ids: list of contract_id strings to test (e.g. ["pdf_2", "pdf_3"]).
                      If None, defaults to the first 5 in the manifest.
    """
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)

    manifest_by_id = {entry["contract_id"]: entry for entry in manifest}

    if contract_ids is None:
        contract_ids = [entry["contract_id"] for entry in manifest[:5]]

    for contract_id in contract_ids:
        print(f"\n{'='*60}")
        print(f"CONTRACT: {contract_id}")
        print('='*60)

        entry = manifest_by_id.get(contract_id)
        if entry is None:
            print(f"  !! Not found in manifest, skipping")
            continue

        text_path = f"{text_dir}/{contract_id}.txt"
        with open(text_path, encoding="utf-8") as f:
            contract_text = f.read()

        truncated_text = contract_text[:max_chars]
        print(f"  Contract length: {len(contract_text)} chars (using {len(truncated_text)})")

        prompt = PROMPT_TEMPLATE.format(
            schema=json.dumps(CLAUSE_SCHEMA_HINT, indent=2),
            contract_text=truncated_text,
        )

        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=4000,
            )
        except Exception as e:
            print(f"  !! API call itself raised an exception: {e}")
            continue

        choice = response.choices[0]
        raw = choice.message.content
        print(f"  finish_reason: {choice.finish_reason}")
        print(f"  output length: {len(raw) if raw else 0} chars")
        print(f"  usage: {getattr(response, 'usage', 'N/A')}")

        if not raw:
            print("  !! EMPTY response content. Nothing more to show.")
            continue

        print(f"  Raw output (first 800 chars):\n{raw[:800]}")

        cleaned = _strip_markdown_fences(raw)
        try:
            parsed = json.loads(cleaned)
            print(f"  \u2713 Parsed successfully: {list(parsed.keys())}")
        except json.JSONDecodeError as e:
            print(f"  !! JSON parse failed: {e}")

### 2.4 Run the pipeline on all 50 contracts

In [ ]:
results = run_pipeline()

### 2.5 Extraction health check

A quick summary of how many contracts returned "not found" or an error for each clause type — a first signal on extraction quality ahead of the formal ground-truth evaluation in Part 3.

In [ ]:
with open("results.json", encoding="utf-8") as f:
    results = json.load(f)

for field in ["termination_clause", "confidentiality_clause", "liability_clause"]:
    not_found = sum(1 for r in results if r[field] in ("Not found in contract", "ERROR", "EMPTY_RESPONSE", "PARSE_ERROR"))
    print(f"{field}: {not_found}/{len(results)} not found / failed")

## Part 3 — Ground-Truth Evaluation

### 3.1 Load CUAD's ground-truth clause annotations

`master_clauses.csv` provides CUAD's expert-annotated ground truth: one row per contract, one column per clause category (41 categories total), each with a companion `-Answer` column giving a binary Yes/No presence flag.

**Limitation:** CUAD's 41 categories do not include a "Confidentiality" clause type. Confidentiality extraction is therefore not scored against ground truth below; it was validated manually via spot-checking individual contract outputs.

In [ ]:
clauses_df = pd.read_csv("master_clauses.csv")
print(clauses_df.shape)
print(list(clauses_df.columns))
print(clauses_df.head(2))

### 3.2 Join the manifest against ground truth

Matches each contract to its CUAD ground-truth row by filename, normalizing case and file extension. Two contracts required a manual filename override due to formatting differences between the CUAD CSV and the downloaded PDFs (an ampersand rendered as an underscore in one case, and a slightly different contract title in the other); both were confirmed by direct lookup against the source CSV.

In [ ]:
FILENAME_OVERRIDES = {
    "MOELIS_CO_03_24_2014-EX-10.19-STRATEGIC ALLIANCE AGREEMENT.PDF":
        "MOELIS&CO_03_24_2014-EX-10.19-STRATEGIC ALLIANCE AGREEMENT.PDF",
    "HarpoonTherapeuticsInc_20200312_10-K_EX-10.18_12051356_EX-10.18_Development Agreement_Option Agreement.pdf":
        "HarpoonTherapeuticsInc_20200312_10-K_EX-10.18_12051356_EX-10.18_Development Agreement.PDF",
}

def load_ground_truth(master_csv="master_clauses.csv", manifest_path="extracted_pdfs/manifest.json"):
    """
    Joins the manifest (contract_id -> source_filename) against CUAD's
    master_clauses.csv on filename (case/extension-insensitive, with known
    overrides applied), returning a dict keyed by contract_id with the
    relevant ground-truth clause presence flags.
    """
    clauses_df = pd.read_csv(master_csv)
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)

    clauses_df["_norm_filename"] = (
        clauses_df["Filename"].str.replace(".pdf", "", case=False, regex=False).str.lower().str.strip()
    )

    ground_truth = {}
    unmatched = []

    for entry in manifest:
        contract_id = entry["contract_id"]
        filename = entry["source_filename"]

        filename = FILENAME_OVERRIDES.get(filename, filename)
        norm_filename = filename.replace(".pdf", "").replace(".PDF", "").lower().strip()

        match = clauses_df[clauses_df["_norm_filename"] == norm_filename]

        if match.empty:
            unmatched.append(entry["source_filename"])
            continue

        row = match.iloc[0]
        ground_truth[contract_id] = {
            "termination_gt": row.get("Termination For Convenience-Answer", "No"),
            "uncapped_liability_gt": row.get("Uncapped Liability-Answer", "No"),
            "cap_on_liability_gt": row.get("Cap On Liability-Answer", "No"),
        }

    print(f"Matched {len(ground_truth)}/{len(manifest)} contracts. Unmatched: {len(unmatched)}")
    if unmatched:
        print("Still unmatched:", unmatched)

    return ground_truth


ground_truth = load_ground_truth()

### 3.3 Ground-truth class distribution

In [ ]:
print(clauses_df["Termination For Convenience-Answer"].value_counts())
print(clauses_df["Uncapped Liability-Answer"].value_counts())
print(clauses_df["Cap On Liability-Answer"].value_counts())

### 3.4 Score clause-presence detection

Evaluation is framed as binary classification: did the model correctly detect whether a clause type exists, regardless of exact wording. Precision, recall, and accuracy are reported per clause type.

**Termination** is scored against CUAD's `Termination For Convenience` category, which is narrower than the assignment's general "conditions under which either party may terminate" framing. This produces high recall but lower precision, since the model correctly surfaces broader termination language (e.g. termination-for-breach) that falls outside this specific CUAD label — a category-granularity mismatch rather than an extraction failure (see Section 3.5).

**Liability** is scored here against `Uncapped Liability`; Section 3.5 re-scores it against `Cap On Liability`, the more accurate ground-truth match for the assignment's general "limitations of liability" framing.

In [ ]:
def score_clause_detection(results_path="results.json", ground_truth=None):
    """
    Compares LLM-extracted clause presence/absence against CUAD ground truth.
    Treats this as a binary classification problem: did the LLM correctly
    detect whether a clause type exists in the contract, regardless of
    exact wording match.
    """
    with open(results_path, encoding="utf-8") as f:
        results = {r["contract_id"]: r for r in json.load(f)}

    def llm_found(value):
        """True if the LLM claims it found a real clause (not a null/error marker)."""
        return value not in ("Not found in contract", "ERROR", "EMPTY_RESPONSE", "PARSE_ERROR", None, "")

    metrics = {}

    for field, gt_key in [
        ("termination_clause", "termination_gt"),
        ("liability_clause", "uncapped_liability_gt"),
    ]:
        tp = fp = tn = fn = 0

        for contract_id, gt in ground_truth.items():
            if contract_id not in results:
                continue

            llm_says_found = llm_found(results[contract_id][field])
            gt_says_exists = gt[gt_key] == "Yes"

            if gt_says_exists and llm_says_found:
                tp += 1
            elif not gt_says_exists and not llm_says_found:
                tn += 1
            elif not gt_says_exists and llm_says_found:
                fp += 1
            elif gt_says_exists and not llm_says_found:
                fn += 1

        total = tp + fp + tn + fn
        accuracy = (tp + tn) / total if total else 0
        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0

        metrics[field] = {
            "true_positive": tp, "true_negative": tn,
            "false_positive": fp, "false_negative": fn,
            "accuracy": round(accuracy, 3),
            "precision": round(precision, 3),
            "recall": round(recall, 3),
        }

    return metrics


metrics = score_clause_detection(ground_truth=ground_truth)
for field, m in metrics.items():
    print(f"\n{field}:")
    for k, v in m.items():
        print(f"  {k}: {v}")

### 3.5 Re-score liability against `Cap On Liability`

Confirms the category-mismatch explanation from Section 3.4 empirically: scoring the same model output against the more semantically appropriate CUAD category yields substantially higher precision, indicating the earlier lower score reflects the ground-truth category chosen rather than a model weakness.

In [ ]:
def score_liability_vs_cap(results_path="results.json", ground_truth=None):
    """
    Re-scores liability_clause detection against CUAD's Cap On Liability
    category instead of Uncapped Liability, since the prompt's general
    "limitations of liability" framing is a closer semantic match to
    capped-liability language than to the narrower "uncapped" category.
    """
    with open(results_path, encoding="utf-8") as f:
        results = {r["contract_id"]: r for r in json.load(f)}

    def llm_found(value):
        return value not in ("Not found in contract", "ERROR", "EMPTY_RESPONSE", "PARSE_ERROR", None, "")

    tp = fp = tn = fn = 0
    for contract_id, gt in ground_truth.items():
        if contract_id not in results:
            continue
        llm_says_found = llm_found(results[contract_id]["liability_clause"])
        gt_says_exists = gt["cap_on_liability_gt"] == "Yes"

        if gt_says_exists and llm_says_found:
            tp += 1
        elif not gt_says_exists and not llm_says_found:
            tn += 1
        elif not gt_says_exists and llm_says_found:
            fp += 1
        else:
            fn += 1

    total = tp + fp + tn + fn
    print(f"precision: {tp/(tp+fp):.3f}, recall: {tp/(tp+fn) if (tp+fn) else 0:.3f}, accuracy: {(tp+tn)/total:.3f}")


score_liability_vs_cap(ground_truth=ground_truth)